In [1]:
import pandas as pd

In [2]:
mode = "Temporal"
df_matched = pd.read_csv(f"{mode}_extracted_templates.csv")

In [3]:
df_questions = pd.read_csv("dynamicqa_converted_templates.csv")

# Group by 'question' and aggregate the 'statement' values into lists
question_dict = df_questions.groupby('question')['statement'].apply(list).to_dict()

In [4]:
# Check: ensure every matched_template exists in question_dict
missing = set(df_matched['matched_template']) - set(question_dict.keys())
if missing:
    raise ValueError(f"❌ The following matched_template(s) are not in question_dict: {missing}")

In [5]:
# Expand the dataframe
expanded_rows = []

for _, row in df_matched.iterrows():
    question = row['matched_template']
    for i, stmt in enumerate(question_dict[question]):  # i goes from 0 → 4
        new_row = row.copy()

        stmt = stmt.replace("[Y]", "_X_")
        stmt = stmt.replace("[X]", row["subj"])
        new_row['statement'] = stmt

        if mode == "Static":
            new_row['mulan_id'] = f"{row['subj_id']}_{row['prop_id']}S_{i}"
        elif mode == "Temporal":
            new_row['mulan_id'] = f"{row['subj_id']}_{row['prop_id']}T_{i}"
        else:
            raise ValueError(f"❌ Invalid mode: {mode}")
        expanded_rows.append(new_row)

expanded_df = pd.DataFrame(expanded_rows)

In [6]:
expanded_df.sample(n=5)

,Unnamed: 0,id,subj,prop,obj,subj_id,prop_id,obj_id,s_pop,o_pop,...,num_edits,context,replace_quality,replace_name,s_pop_new,o_pop_new,matched_template,extracted_sub,statement,mulan_id
1386,7872,2984189,Hamilton,capital of,South Lanarkshire,Q4131,P1376,Q209142,4828,4309,...,2.0,Hamilton (Scots: Hamiltoun; Scottish Gaelic: ...,0.874378,Dumfries and Galloway,5029.0,4399.0,What is [X] the capital of?,"('Hamilton',)",Hamilton serves as the capital of _X_.,Q4131_P1376T_1
99,155,3049219,Thandiswa Mazwai,occupation,songwriter,Q4348940,P106,Q753110,1704,25550,...,2.0,Thandiswa Nyameka Mazwai (born 31 March 1976)...,0.896252,singer-songwriter,1594.0,2.0,What is [X]'s occupation?,"('Thandiswa Mazwai',)","By occupation, Thandiswa Mazwai is a _X_.",Q4348940_P106T_4
333,1122,6210567,Wonderful World,genre,comedy-drama,Q8031851,P136,Q859369,570,36819,...,2.0,Wonderful World is a 2010 dark [ENTITY] film ...,0.957594,LGBT-related film,245.0,1.0,What genre is [X]?,"('Wonderful World',)",Wonderful World is categorized in the genre _X_.,Q8031851_P136T_2
85,136,4918912,Larry Coon,occupation,computer scientist,Q6490123,P106,Q82594,199,11699,...,2.0,Larry Coon is a [ENTITY]. The New York Times ...,0.890973,programmer,332.0,3.0,What is [X]'s occupation?,"('Larry Coon',)",Larry Coon works as a _X_.,Q6490123_P106T_2
373,1268,2282388,Decay,genre,horror film,Q3020975,P136,Q200092,729,90942,...,2.0,Decay is a 2012 [ENTITY] by Luke Thompson (of...,0.974067,science fiction film,3277.0,15.0,What genre is [X]?,"('Decay',)",The genre associated with Decay is _X_.,Q3020975_P136T_3


In [7]:
expanded_df.to_csv(f"{mode}_DQA_statments.csv", index=False)